# Multimodal Fake News Detection: Four Variants

This notebook runs in Google Colab or Kaggle. It trains and evaluates text-only, image-only, multimodal-concat, and multimodal-cross-attention classifiers on the verified Fakeddit subset.

**Fast mode is enabled by default** so a demonstration finishes quickly. Set `FAST_MODE = False` after confirming the pipeline works. Upload or mount a folder containing `fakeddit_verified.csv` and an `images/` directory, then update the two paths in the configuration cell.

In [ ]:
!pip -q install -U transformers accelerate scikit-learn pandas pillow matplotlib seaborn tqdm tabulate

In [ ]:
import os, json, random, zipfile, shutil, time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoImageProcessor, ViTModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
FAST_MODE = True
MAX_SAMPLES = 256 if FAST_MODE else None
EPOCHS = 2 if FAST_MODE else 10
BATCH_SIZE = 32
MAX_LENGTH = 128
TEXT_CHECKPOINT = 'roberta-base'
IMAGE_CHECKPOINT = 'google/vit-base-patch16-224'

# Colab example: '/content/data/raw/fakeddit_verified.csv'
# Kaggle example: '/kaggle/input/fakeddit-verified/fakeddit_verified.csv'
DATA_CSV = '/content/data/raw/fakeddit_verified.csv'
IMAGE_DIR = '/content/data/raw/images'
OUTPUT_DIR = Path('outputs_four_variants')
OUTPUT_DIR.mkdir(exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Set DATA_CSV and IMAGE_DIR before continuing if the automatic paths do not exist.')

## Data setup

Upload `fakeddit_verified.csv` and the matching `images/` folder to Colab/Kaggle. The CSV must contain `id`, `title`, `image_url`, and `label`, where `image_url` stores local filenames such as `abc123.jpg`. This notebook never downloads external image URLs during training.

In [ ]:
csv_path = Path(DATA_CSV); image_root = Path(IMAGE_DIR)
if not csv_path.exists() or not image_root.exists():
    print('Dataset not found at the configured paths.')
    print('Upload/mount the verified CSV and images, then edit DATA_CSV and IMAGE_DIR and rerun this cell.')
else:
    df = pd.read_csv(csv_path)
    required = {'id','title','image_url','label'}
    missing = required - set(df.columns)
    if missing: raise ValueError(f'Missing columns: {sorted(missing)}')
    df['label'] = df['label'].astype(int)
    df['local_image'] = df['image_url'].astype(str).map(lambda x: image_root / Path(x).name)
    before = len(df)
    df = df[df['title'].notna() & df['title'].astype(str).str.strip().ne('')]
    df = df[df['label'].isin([0,1])]
    df = df[df['local_image'].map(lambda p: p.is_file() and p.stat().st_size > 0)]
    df = df.drop_duplicates('id').drop_duplicates(['title','image_url']).reset_index(drop=True)
    if MAX_SAMPLES and len(df) > MAX_SAMPLES:
        df, _ = train_test_split(df, train_size=MAX_SAMPLES, stratify=df['label'], random_state=SEED)
        df = df.reset_index(drop=True)
    train_df, rest_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
    val_df, test_df = train_test_split(rest_df, test_size=0.50, stratify=rest_df['label'], random_state=SEED)
    print(f'Usable rows: {len(df)} (filtered from {before})')
    print('Splits:', len(train_df), len(val_df), len(test_df))
    print('Labels:', df['label'].value_counts().to_dict())
    assert not (set(train_df.id) & set(val_df.id) or set(train_df.id) & set(test_df.id) or set(val_df.id) & set(test_df.id))

In [ ]:
if 'df' not in globals():
    raise RuntimeError('Configure DATA_CSV and IMAGE_DIR, then rerun the data setup cell.')
tokenizer = AutoTokenizer.from_pretrained(TEXT_CHECKPOINT)
text_model = AutoModel.from_pretrained(TEXT_CHECKPOINT).to(DEVICE).eval()
image_processor = AutoImageProcessor.from_pretrained(IMAGE_CHECKPOINT)
image_model = ViTModel.from_pretrained(IMAGE_CHECKPOINT).to(DEVICE).eval()
for model in (text_model, image_model):
    for p in model.parameters(): p.requires_grad = False

@torch.no_grad()
def extract_features(frame):
    text_features, image_features, labels, ids = [], [], [], []
    for start in range(0, len(frame), BATCH_SIZE):
        part = frame.iloc[start:start+BATCH_SIZE]
        texts = part['title'].astype(str).tolist()
        tok = tokenizer(texts, padding='max_length', truncation=True, max_length=MAX_LENGTH, return_tensors='pt').to(DEVICE)
        t = text_model(**tok).last_hidden_state[:, 0, :]
        images = [Image.open(p).convert('RGB') for p in part['local_image']]
        pix = image_processor(images=images, return_tensors='pt')['pixel_values'].to(DEVICE)
        v = image_model(pixel_values=pix).pooler_output
        text_features.append(t.cpu()); image_features.append(v.cpu())
        labels.extend(part['label'].tolist()); ids.extend(part['id'].tolist())
    return torch.cat(text_features), torch.cat(image_features), torch.tensor(labels), ids

cache = {}
for name, frame in [('train', train_df), ('val', val_df), ('test', test_df)]:
    cache[name] = extract_features(frame)
    print(name, cache[name][0].shape, cache[name][1].shape)
torch.save(cache, OUTPUT_DIR / 'frozen_features.pt')
print('Cached encoder features:', OUTPUT_DIR / 'frozen_features.pt')

In [ ]:
class VariantClassifier(nn.Module):
    def __init__(self, variant, hidden=256):
        super().__init__(); self.variant = variant
        self.text_proj = nn.Linear(768, hidden); self.image_proj = nn.Linear(768, hidden)
        if variant == 'cross_attention':
            self.cross = nn.MultiheadAttention(hidden, 8, batch_first=True)
        dims = {'text_only': 768, 'image_only': 768, 'concat': hidden*2, 'cross_attention': hidden*2}
        self.head = nn.Sequential(nn.Linear(dims[variant], hidden), nn.ReLU(), nn.Dropout(0.1), nn.Linear(hidden, 2))
    def forward(self, text, image):
        if self.variant == 'text_only': return self.head(text)
        if self.variant == 'image_only': return self.head(image)
        t, v = self.text_proj(text), self.image_proj(image)
        if self.variant == 'cross_attention':
            t_seq, v_seq = t[:, None, :], v[:, None, :]
            t = self.cross(t_seq, v_seq, v_seq)[0].squeeze(1)
            v = self.cross(v_seq, t_seq, t_seq)[0].squeeze(1)
        return self.head(torch.cat([t, v], dim=1))

def metrics(y, pred, prob):
    p, r, f, _ = precision_recall_fscore_support(y, pred, average='macro', zero_division=0)
    return {'accuracy': accuracy_score(y, pred), 'precision_macro': p, 'recall_macro': r,
            'f1_macro': f, 'roc_auc': roc_auc_score(y, prob) if len(np.unique(y)) == 2 else np.nan}

def run_variant(name):
    model = VariantClassifier(name).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss()
    train = TensorDataset(cache['train'][0], cache['train'][1], cache['train'][2])
    val = TensorDataset(cache['val'][0], cache['val'][1], cache['val'][2])
    test = TensorDataset(cache['test'][0], cache['test'][1], cache['test'][2])
    train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
    best, patience, best_state = float('inf'), 0, None
    history = []
    for epoch in range(EPOCHS):
        model.train(); total = 0
        for text, image, y in train_loader:
            text, image, y = text.to(DEVICE), image.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); loss = loss_fn(model(text, image), y); loss.backward(); opt.step(); total += loss.item()
        model.eval(); val_logits, val_y = [], []
        with torch.no_grad():
            for text, image, y in DataLoader(val, batch_size=BATCH_SIZE):
                val_logits.append(model(text.to(DEVICE), image.to(DEVICE)).cpu()); val_y.append(y)
        logits, y = torch.cat(val_logits), torch.cat(val_y); val_loss = loss_fn(logits, y).item()
        val_pred = logits.argmax(1).numpy(); val_prob = logits.softmax(1)[:,1].numpy()
        row = {'variant': name, 'epoch': epoch+1, 'train_loss': total/len(train_loader), 'val_loss': val_loss, **metrics(y.numpy(), val_pred, val_prob)}
        history.append(row); print(row)
        if val_loss < best: best, patience, best_state = val_loss, 0, {k:v.cpu().clone() for k,v in model.state_dict().items()}
        else: patience += 1
        if patience >= 3: break
    model.load_state_dict(best_state); model.eval(); logits, y = [], []
    with torch.no_grad():
        for text, image, target in DataLoader(test, batch_size=BATCH_SIZE):
            logits.append(model(text.to(DEVICE), image.to(DEVICE)).cpu()); y.append(target)
    logits, y = torch.cat(logits), torch.cat(y); pred = logits.argmax(1).numpy(); prob = logits.softmax(1)[:,1].numpy()
    result = {'variant': name, **metrics(y.numpy(), pred, prob)}
    torch.save({'variant': name, 'state_dict': model.state_dict(), 'result': result}, OUTPUT_DIR / f'{name}.pt')
    pd.DataFrame(history).to_csv(OUTPUT_DIR / f'{name}_history.csv', index=False)
    cm = confusion_matrix(y.numpy(), pred); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues'); plt.title(name); plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.savefig(OUTPUT_DIR / f'{name}_confusion_matrix.png'); plt.close()
    return result

results = [run_variant(name) for name in ['text_only','image_only','concat','cross_attention']]
results_df = pd.DataFrame(results); results_df.to_csv(OUTPUT_DIR / 'comparison.csv', index=False)
results_df.to_markdown(OUTPUT_DIR / 'comparison.md', index=False)
display(results_df)
print('Saved outputs in', OUTPUT_DIR.resolve())

## Interpretation and limitations

The table compares all four variants on the same held-out split. In fast mode, the result is a pipeline demonstration rather than a publishable result. For research numbers, set `FAST_MODE = False`, use the complete verified dataset, repeat with multiple seeds, and report mean and standard deviation. These models classify patterns associated with the labels; they do not prove that a claim is factually true. Evidence retrieval, source links, and human review are required for factual verification.